In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

RANDOM_SEED = 123
THINNING_HOURS = 6

In [2]:
# This allows the notebook to run from the repository root
# or from data_labeling/eda/.

current_directory = Path.cwd().resolve()

ROOT = None
for candidate in [current_directory, *current_directory.parents]:
    if (candidate / "data_labeling").exists() and (candidate / "modeling").exists():
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError("Could not find the repository root.")

STAGE1_TRAIN_CSV = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "CL_Labels"
    / "Stage1_train.csv"
)

OUTPUT_DIRECTORY = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "replay_buffers"
)

OUTPUT_CSV = OUTPUT_DIRECTORY / f"stage1_fl_replay_{BUFFER_SIZE}.csv"
OUTPUT_METADATA = OUTPUT_DIRECTORY / f"stage1_fl_replay_{BUFFER_SIZE}_metadata.json"

print("Repository:", ROOT)
print("Input CSV:", STAGE1_TRAIN_CSV)
print("Output CSV:", OUTPUT_CSV)

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Input CSV: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Stage1_train.csv
Output CSV: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/replay_buffers/stage1_fl_replay_1500.csv


In [3]:
stage1 = pd.read_csv(STAGE1_TRAIN_CSV)

required_columns = {"label", "goes_class"}
missing_columns = required_columns.difference(stage1.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

if not set(stage1["goes_class"].unique()).issubset({0, 1}):
    raise ValueError("goes_class must contain only 0 and 1.")

print("Total Stage 1 training samples:", len(stage1))
print()
print(stage1["goes_class"].value_counts().sort_index())

Total Stage 1 training samples: 46638

goes_class
0    38015
1     8623
Name: count, dtype: int64


Extract the image timestamps

In [4]:
# Example filename:
# 2015/01/01/HMI.m2015.01.01_00.00.00.jpg

timestamp_text = stage1["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})",
    expand=False,
)

stage1["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
    errors="coerce",
)

missing_timestamps = stage1["timestamp"].isna().sum()

print("Missing timestamps:", missing_timestamps)
print("First timestamp:", stage1["timestamp"].min())
print("Last timestamp:", stage1["timestamp"].max())

if missing_timestamps > 0:
    display(stage1.loc[stage1["timestamp"].isna(), ["label"]].head(20))
    raise ValueError("Some timestamps could not be parsed.")

Missing timestamps: 0
First timestamp: 2010-12-06 07:00:00
Last timestamp: 2017-01-06 03:00:00


keep only Stage 1 flare samples

In [5]:
stage1_fl = stage1.loc[stage1["goes_class"] == 1].copy()

stage1_fl["year"] = stage1_fl["timestamp"].dt.year
stage1_fl["month"] = stage1_fl["timestamp"].dt.month

print("Available Stage 1 FL samples:", len(stage1_fl))
print()
print("FL samples by year:")
display(stage1_fl.groupby("year").size().rename("available_fl"))

Available Stage 1 FL samples: 8623

FL samples by year:


year
2011    1235
2012    1358
2013    1415
2014    2702
2015    1663
2016     250
Name: available_fl, dtype: int64

keeps at most one randomly selected FL image from every six-hour window.

In [6]:
stage1_fl["time_window"] = stage1_fl["timestamp"].dt.floor(
    f"{THINNING_HOURS}h"
)

# Shuffle first so the selected image inside each window is not
# always the earliest image.
shuffled_fl = stage1_fl.sample(
    frac=1.0,
    random_state=RANDOM_SEED,
)

thinned_fl = (
    shuffled_fl
    .drop_duplicates(subset="time_window", keep="first")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("FL images before thinning:", len(stage1_fl))
print("FL images after thinning:", len(thinned_fl))
print("Removed near-duplicate images:", len(stage1_fl) - len(thinned_fl))

if len(thinned_fl) < BUFFER_SIZE:
    raise ValueError(
        f"Only {len(thinned_fl)} samples remain after thinning, "
        f"but BUFFER_SIZE={BUFFER_SIZE}. "
        "Reduce THINNING_HOURS to 3 and rerun."
    )

FL images before thinning: 8623
FL images after thinning: 1655
Removed near-duplicate images: 6968


Select the fixed replay buffer

In [7]:
BUFFER_SIZE = 1655

Select the fixed replay buffer

In [8]:
replay_buffer = (
    thinned_fl
    .sample(
        n=BUFFER_SIZE,
        replace=False,
        random_state=RANDOM_SEED,
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

replay_buffer["selection_reason"] = (
    f"stage1_fl_temporal_thinning_{THINNING_HOURS}h"
)

# Retain only the columns needed for training and auditing.
replay_buffer = replay_buffer[
    [
        "label",
        "goes_class",
        "timestamp",
        "year",
        "month",
        "time_window",
        "selection_reason",
    ]
]

print("Selected replay samples:", len(replay_buffer))
print("Unique image paths:", replay_buffer["label"].nunique())
print("Unique time windows:", replay_buffer["time_window"].nunique())
print("Labels:", replay_buffer["goes_class"].value_counts().to_dict())

Selected replay samples: 1655
Unique image paths: 1655
Unique time windows: 1655
Labels: {1: 1655}


Inspect the temporal distribution

In [9]:
year_distribution = (
    replay_buffer
    .groupby("year")
    .size()
    .rename("replay_fl")
    .to_frame()
)

year_distribution["percentage"] = (
    100 * year_distribution["replay_fl"] / len(replay_buffer)
)

display(year_distribution)

month_distribution = (
    replay_buffer
    .groupby(["year", "month"])
    .size()
    .rename("replay_fl")
    .reset_index()
)

display(month_distribution)

,replay_fl,percentage
year,,
2011,242,14.622356
2012,263,15.891239
2013,278,16.797583
2014,507,30.634441
2015,318,19.214502
2016,47,2.839879


,year,month,replay_fl
0,2011,1,4
1,2011,2,18
2,2011,3,43
3,2011,5,7
4,2011,6,10
...,...,...,...
56,2016,1,4
57,2016,2,17
58,2016,4,5
59,2016,7,16


Save the buffer and metadata

In [11]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

replay_buffer.to_csv(OUTPUT_CSV, index=False)

metadata = {
    "source_csv": str(STAGE1_TRAIN_CSV.relative_to(ROOT)),
    "output_csv": str(OUTPUT_CSV.relative_to(ROOT)),
    "selection": "stage1_fl_only_temporally_thinned_random_selection",
    "random_seed": RANDOM_SEED,
    "requested_buffer_size": BUFFER_SIZE,
    "actual_buffer_size": len(replay_buffer),
    "thinning_hours": THINNING_HOURS,
    "available_stage1_fl_before_thinning": len(stage1_fl),
    "available_stage1_fl_after_thinning": len(thinned_fl),
    "class_counts": {
        "NF": int((replay_buffer["goes_class"] == 0).sum()),
        "FL": int((replay_buffer["goes_class"] == 1).sum()),
    },
    "year_counts": {
        str(year): int(count)
        for year, count in replay_buffer.groupby("year").size().items()
    },
}

with OUTPUT_METADATA.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

csv_sha256 = hashlib.sha256(OUTPUT_CSV.read_bytes()).hexdigest()

print("Saved CSV:", OUTPUT_CSV)
print("Saved metadata:", OUTPUT_METADATA)
print("CSV SHA-256:", csv_sha256)

Saved CSV: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/replay_buffers/stage1_fl_replay_1500.csv
Saved metadata: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/replay_buffers/stage1_fl_replay_1500_metadata.json
CSV SHA-256: 01ebb6c9beadc27b2115e4f1f99c6cfdc2301895cc3726450f630cee296fdc16
